# Sensory landscapes

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

An arena on its own is empty space. What makes it an *assay* is what is spread over it : a plume of
odor coming off a source, a temperature gradient across the plate, a wind that pushes from one
direction and gusts from another. Larvaworld calls these **sensory landscapes**, and a larva senses
them through the corresponding brain module - the olfactor, the thermosensor, the windsensor.

There are three, and they behave differently :

| landscape | field | what a larva senses |
|---|---|---|
| **Odorscape** | `odorscape` | odor concentration, and its change along the path |
| **Thermoscape** | `thermoscape` | local warming and cooling relative to the plate temperature |
| **Windscape** | `windscape` | wind direction and speed, including discrete air puffs |

This notebook builds one of each and renders it, so you can see the field the agent is in rather
than infer it from numbers.

**What you will be able to do afterwards**

- Choose between a Gaussian and a diffusion odorscape and say what changes.
- Attach the matching kind of `Odor` to a source.
- Say what a thermoscape exposes to configuration and what it does not.
- Configure a steady wind, and schedule single or repetitive air puffs.
- Render any of them, optionally animating the field over time.

**Prerequisites** : [Building an environment](environment_configuration.ipynb).

**Cost** : seconds to build. Rendering is off by default because it opens a window.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_SINGLE_DEMO` | `False` | rendering one landscape |
| `RUN_ALL_COMBINATIONS` | `False` | rendering every combination of odor, puff and wind mode |
| `SAVE_MEDIA` | `False` | writing the renders to video files |

## Setup

In [1]:
%matplotlib inline

import numpy as np

import larvaworld
from larvaworld.lib import reg, util
from larvaworld.lib.param.composition import Odor

larvaworld.VERBOSE = 1

# Tutorial safety switches
RUN_SINGLE_DEMO = False  # render one landscape
RUN_ALL_COMBINATIONS = False  # render every combination (slow)
SAVE_MEDIA = False  # write videos
MEDIA_DIR = "./media"

Initializing larvaworld registry


Registry configured!


## Section 1 : Odorscapes

An odorscape decides how the concentration emitted by a source spreads over the arena. Larvaworld
offers two, and the choice is a modeling decision, not a cosmetic one :

- **Gaussian** (`GaussianValueLayer`) - the concentration is an analytical function of the distance
  to each source. It is static, exact and cheap, and it is the right choice when you care about the
  gradient the animal climbs rather than about plume dynamics.
- **Diffusion** (`DiffusionValueLayer`) - the concentration is a grid that is diffused and
  evaporated at every timestep. It is dynamic and can be advected by wind, at the cost of
  computation. Use it when the *history* of the odor field matters.

The odor attached to a source has to match : `Odor.oG` builds a Gaussian odor, `Odor.oD` a
diffusion one.

In [2]:
def get_odorscape(m):
    """Return the food_params/odorscape pair for a two-source arena.

    Two sources are placed 2 cm apart, left and right of the centre, each emitting
    its own odor identity so that a larva can in principle tell them apart.
    """
    if m == "Diffusion":
        oS = reg.gen.DiffusionValueLayer(gaussian_sigma=(0.95, 0.5), evap_const=0.9)
        oR = Odor.oD(id="Odor_R")
        oL = Odor.oD(id="Odor_L")
    elif m == "Gaussian":
        oS = reg.gen.GaussianValueLayer()
        oR = Odor.oG(id="Odor_R")
        oL = Odor.oG(id="Odor_L")
    else:
        raise ValueError(f"Unknown odorscape mode : {m!r}")

    kws = {"group": "Source", "radius": 0.003, "amount": 0.0}

    sus = {
        **reg.gen.Food(
            unique_id="Source_L", c="blue", odor=oL, pos=(-0.01, 0.0), **kws
        ).entry(),
        **reg.gen.Food(
            unique_id="Source_R", c="cyan", odor=oR, pos=(0.01, 0.0), **kws
        ).entry(),
    }

    return util.AttrDict(
        {"food_params": reg.gen.FoodConf(source_units=sus), "odorscape": oS}
    )

In [3]:
gaussian = get_odorscape("Gaussian")
diffusion = get_odorscape("Diffusion")

print("Gaussian  :", gaussian.odorscape.__class__.__name__)
print(
    "Diffusion :",
    diffusion.odorscape.__class__.__name__,
    f"(sigma={diffusion.odorscape.gaussian_sigma}, evaporation={diffusion.odorscape.evap_const})",
)
print("Sources   :", gaussian.food_params.source_units.keylist)

Gaussian  : GaussianValueLayerUnit
Diffusion : DiffusionValueLayerUnit (sigma=(0.95, 0.5), evaporation=0.9)
Sources   : ['Source_L', 'Source_R']


## Section 2 : Windscapes

A windscape has a steady component - a direction and a speed - and a set of **air puffs** :
discrete gusts with their own direction, speed, duration and start time. Puffs are how the
mechanosensory startle assays are reproduced, in which a larva is hit by a defined burst of air and
its escape response is measured.

Three arrangements are built below :

- `"single"` - a sequence of puffs, each from a different direction, ten seconds apart,
- `"repetitive"` - one puff group repeating at a fixed interval from a fixed direction,
- `"no"` - no puffs at all, just a steady wind.

In [4]:
def get_windscape(m):
    """Return the windscape/border pair for one puff arrangement."""
    kws = {"duration": 5, "speed": 50}
    Npuffs = 10

    if m == "single":
        puffs = {
            i: reg.gen.AirPuff(
                direction=i / Npuffs * 2 * np.pi, start_time=5 + 10 * i, **kws
            ).nestedConf
            for i in range(Npuffs)
        }
        ws = 0.0
    elif m == "repetitive":
        puffs = {
            "puff_group": reg.gen.AirPuff(
                direction=np.pi / 4, start_time=5, N=Npuffs, interval=10.0, **kws
            ).nestedConf
        }
        ws = 0.0
    elif m == "no":
        puffs = {}
        ws = 10.0
    else:
        raise ValueError(f"Unknown puff mode : {m!r}")

    wS = reg.gen.WindScape(wind_direction=0.0, wind_speed=ws, puffs=puffs)
    return util.AttrDict({"windscape": wS, "border_list": {}})

In [5]:
for mode in ["single", "repetitive", "no"]:
    w = get_windscape(mode).windscape
    print(f"{mode:12s} steady speed={w.wind_speed:5.1f}  puffs={len(w.puffs)}")

single       steady speed=  0.0  puffs=10
repetitive   steady speed=  0.0  puffs=1
no           steady speed= 10.0  puffs=0


## Section 3 : Thermoscapes

A thermoscape is a plate at a baseline temperature with a number of hot and cold sources on it,
each contributing a Gaussian bump. What a larva senses at a position is not the temperature itself
but the *warming* and *cooling* relative to the plate - the two channels its thermosensor reads.

Its configuration is worth reading carefully, because it is the one landscape whose configurable
surface is smaller than its runtime one. What an environment can set is the grid :

In [6]:
thermo = reg.gen.ThermoScape()

print("Configurable fields :")
thermo.nestedConf.print()

Configurable fields :
     color : white
     fixed_max : False
     grid_dims : (51, 51)
     initial_value : 0.0
     unique_id : ThermoScape


The source layout - where the hot and cold spots are, and how strong they are - is **not** part of
the environment configuration. It comes from the defaults of the runtime `ThermoScape` class, which
the simulation instantiates with whatever the configuration provides :

| runtime default | value |
|---|---|
| plate temperature | 22 C |
| source positions (relative arena coordinates) | `[0.5, 0.05]`, `[0.05, 0.5]`, `[0.5, 0.95]`, `[0.95, 0.5]` |
| temperature deltas | `+8`, `-8`, `+8`, `-8` C |

That is the four-source alternating layout used in thermotaxis assays. Changing it currently means
constructing `larvaworld.lib.model.envs.valuegrid.ThermoScape` directly rather than going through
the environment configuration - worth knowing before you plan an experiment around a custom
temperature landscape.

In [7]:
from larvaworld.lib.model.envs.valuegrid import ThermoScape

runtime_thermo = ThermoScape()
print(f"plate temperature : {runtime_thermo.plate_temp} C")
print(f"sources           : {runtime_thermo.thermo_sources}")
print(f"deltas            : {runtime_thermo.thermo_source_dTemps} C")
print()
print("Sensed at the plate centre :", runtime_thermo.get_value((0.5, 0.5)))

plate temperature : 22 C
sources           : {'0': [0.5, 0.05], '1': [0.05, 0.5], '2': [0.5, 0.95], '3': [0.95, 0.5]}
deltas            : {'0': 8, '1': -8, '2': 8, '3': -8} C

Sensed at the plate centre : {'cool': 5.81295310974418, 'warm': 5.81295310974418}


## Section 4 : Assembling and rendering

An environment is the arena plus whichever landscapes the assay needs. `EnvConf.visualize` runs a
simulation with no agents in it and renders the field, which is the only reliable way to check that
a landscape looks like what you intended.

The `func` argument is the interesting part : it is called on the model at every step, so the field
can be driven over time. The two functions below rotate the wind direction and ramp its speed, which
is how you would build a slowly turning wind stimulus.

In [8]:
def get_env(Om=None, Pm=None):
    """Build an environment with the requested odorscape and puff arrangement."""
    dO = get_odorscape(m=Om) if Om is not None else {}
    dW = get_windscape(m=Pm) if Pm is not None else {}
    return reg.gen.Env(**dO, **dW)


def get_id(Om=None, Pm=None, Wm=None):
    """A filename describing the combination being rendered."""
    if Om is not None and Pm is None:
        return f"{Om}_odorscape"
    if Om is None and Pm is not None:
        return f"{Pm}_air-puffs_variable_wind_{Wm}"
    return f"{Om}_odorscape_{Pm}_air-puffs_variable_wind_{Wm}"

In [9]:
env_gaussian = get_env(Om="Gaussian")
env_windy = get_env(Om="Diffusion", Pm="repetitive")

print(
    "odor only  :",
    env_gaussian.odorscape.__class__.__name__,
    "| windscape :",
    env_gaussian.windscape,
)
print(
    "odor+wind  :",
    env_windy.odorscape.__class__.__name__,
    "| windscape :",
    env_windy.windscape.__class__.__name__,
)

odor only  : GaussianValueLayerUnit | windscape : None
odor+wind  : DiffusionValueLayerUnit | windscape : WindScapeUnit


In [10]:
def run_scape(Om=None, Pm=None, Wm=None, duration=0.15, **kwargs):
    """Render one landscape combination, optionally driving the wind over time."""

    def rotate_wind(model):
        model.windscape.set_wind_direction((model.t / 10 / np.pi) % (2 * np.pi))

    def ramp_wind(model):
        model.windscape.wind_speed = model.t % 100

    func = {"direction": rotate_wind, "speed": ramp_wind}.get(Wm)

    id = get_id(Om=Om, Pm=Pm, Wm=Wm)
    get_env(Om=Om, Pm=Pm).visualize(
        duration=duration,
        id=id,
        screen_kws={
            "vis_mode": "video",
            "save_video": SAVE_MEDIA,
            "media_dir": MEDIA_DIR,
            "video_file": id,
        },
        func=func,
        **kwargs,
    )

In [11]:
if RUN_SINGLE_DEMO:
    run_scape(Om="Diffusion", Pm="repetitive", Wm="speed", duration=2)
else:
    print(
        "Set RUN_SINGLE_DEMO = True to render a diffusion odorscape under a ramping wind."
    )

Set RUN_SINGLE_DEMO = True to render a diffusion odorscape under a ramping wind.


Rendering every combination is useful once, as a visual catalog of what the three landscapes can
do. It is slow - one short simulation per combination - so it is off by default.

In [12]:
if RUN_ALL_COMBINATIONS:
    for Om in ["Gaussian", "Diffusion", None]:
        for Pm in [None, "single", "repetitive", "no"]:
            for Wm in [None, "direction", "speed", "no"]:
                if (Pm is None) != (Wm is None):
                    continue
                if Om is None and Pm is None:
                    continue
                run_scape(Om=Om, Pm=Pm, Wm=Wm)
else:
    print("Set RUN_ALL_COMBINATIONS = True to render the full catalog.")

Set RUN_ALL_COMBINATIONS = True to render the full catalog.

## Where to go next

- [Building an environment](environment_configuration.ipynb) - the arena these landscapes sit on.
- [Custom brain modules](../6_extending_larvaworld/custom_brain_modules.ipynb) - the sensor side :
  how a larva reads a landscape, and how to replace that reading with your own.
- Reference : [Arenas and substrates](../../agents_environments/arenas_and_substrates.md) and
  [Brain module architecture](../../agents_environments/brain_module_architecture.md).